# RAG Agent validation (dev)
Este notebook es utilizado para realizar la validación del RAG antes de hacer el deploy en el serving endpoint

> Nota: en Compute Serverless suele faltar `mlflow`; por eso instalamos dependencias en la primera celda.


In [0]:
# # # # ==============================
# # # # 1) Dependencias (solo para pruebas locales)
# # # # ==============================
# # # # En Serverless compute puede no venir mlflow instalado.
# # # %pip install -U mlflow
# # # %pip install -U databricks-vectorsearch mlflow databricks-sdk
# dbutils.library.restartPython()


In [0]:
# ==============================
# 2) Config (ajustar a tu workspace)
# ==============================
from pathlib import Path
import time

# --- Vector Search ---
VS_ENDPOINT = "bind_agent_vs"
VS_INDEX_FULL_NAME = "bind_agent.docs.pdf_chunks_vs_idx"

# --- Endpoints ---
EMBED_ENDPOINT = "databricks-bge-large-en"  # for query_vector
EMB_ENDPOINT = EMBED_ENDPOINT  # backward-compatible alias
# LLM_ENDPOINT = "databricks-gemma-3-12b"
LLM_ENDPOINT = 'databricks-llama-4-maverick' 
# LLM_ENDPOINT = _get_env("RAG_LLM_ENDPOINT", default="databricks-gpt-5-2")

# --- Retrieval params ---
TOP_K_CANDIDATES = "40"
LEX_FALLBACK_LIMIT = "20"
# TOP_K_CANDIDATES = "120"
# LEX_FALLBACK_LIMIT = "60"
TOP_K_FINAL = "8"

# --- Prompt/context limits ---
MAX_CONTEXT_CHARS = "14000"
RERANK_SNIPPET_CHARS = "1200"

# --- LLM params ---
TEMPERATURE_RERANK = "0.0"
TEMPERATURE_ANSWER = "0.2"
MAX_TOKENS_ANSWER = "900"

# --- Retry params ---
MAX_RETRIES = "3"
RETRY_SLEEP_SECS = "1.0"

# MLflow / UC
# Recomendación: catalog.schema.model
UC_MODEL_NAME = "bind_agent.docs.rag_agent"

# Experimento: usar ruta en /Users/... para evitar problemas de permisos
current_user = spark.sql("select current_user() as u").first()["u"]
EXPERIMENT_PATH = f"/Users/{current_user}/bind_agent/rag_agent_deploy"

# Serving endpoint para el agente
MODEL_SERVING_ENDPOINT = "bind_agent_rag_agent"


In [0]:
import os, importlib.util
from pathlib import Path

# 0) Setear env vars usadas en el RAG
os.environ["RAG_LLM_ENDPOINT"] = LLM_ENDPOINT
os.environ["RAG_EMBED_ENDPOINT"] = EMBED_ENDPOINT
os.environ["RAG_VS_ENDPOINT"] = VS_ENDPOINT
os.environ["RAG_VS_INDEX"] = VS_INDEX_FULL_NAME
os.environ["RAG_VS_INDEX_FULL_NAME"] = VS_INDEX_FULL_NAME
os.environ["RAG_TOP_K_CANDIDATES"] = TOP_K_CANDIDATES
os.environ["RAG_TOP_K_FINAL"] = TOP_K_FINAL
os.environ["RAG_LEX_FALLBACK_LIMIT"] = LEX_FALLBACK_LIMIT
os.environ["RAG_MAX_CONTEXT_CHARS"] = MAX_CONTEXT_CHARS
os.environ["RAG_RERANK_SNIPPET_CHARS"] = RERANK_SNIPPET_CHARS
os.environ["RAG_TEMPERATURE_RERANK"] = TEMPERATURE_RERANK
os.environ["RAG_TEMPERATURE_ANSWER"] = TEMPERATURE_ANSWER
os.environ["RAG_MAX_TOKENS_ANSWER"] = MAX_TOKENS_ANSWER
os.environ["RAG_MAX_RETRIES"] = MAX_RETRIES
os.environ["RAG_RETRY_SLEEP_SECS"] = RETRY_SLEEP_SECS
# os.environ["DATABRICKS_TOKEN"] = token # Acceso para el service endpoint al search index 
# os.environ["DATABRICKS_HOST"] = host # Acceso para el service endpoint al search index 

# env usadas en el serving endpoint
env_vars = {
    "RAG_LLM_ENDPOINT": os.environ["RAG_LLM_ENDPOINT"],
    "RAG_EMBED_ENDPOINT": os.environ["RAG_EMBED_ENDPOINT"],
    "RAG_VS_ENDPOINT": os.environ["RAG_VS_ENDPOINT"],
    "RAG_VS_INDEX": os.environ["RAG_VS_INDEX"],
    "RAG_VS_INDEX_FULL_NAME": os.environ["RAG_VS_INDEX_FULL_NAME"],
    "RAG_TOP_K_CANDIDATES": os.environ["RAG_TOP_K_CANDIDATES"],
    "RAG_TOP_K_FINAL": os.environ["RAG_TOP_K_FINAL"],
    "RAG_LEX_FALLBACK_LIMIT": os.environ["RAG_LEX_FALLBACK_LIMIT"],
    "RAG_MAX_CONTEXT_CHARS": os.environ["RAG_MAX_CONTEXT_CHARS"],
    "RAG_RERANK_SNIPPET_CHARS": os.environ["RAG_RERANK_SNIPPET_CHARS"],
    "RAG_TEMPERATURE_RERANK": os.environ["RAG_TEMPERATURE_RERANK"],
    "RAG_TEMPERATURE_ANSWER": os.environ["RAG_TEMPERATURE_ANSWER"],
    "RAG_MAX_TOKENS_ANSWER": os.environ["RAG_MAX_TOKENS_ANSWER"],
    "RAG_MAX_RETRIES": os.environ["RAG_MAX_RETRIES"],
    "RAG_RETRY_SLEEP_SECS": os.environ["RAG_RETRY_SLEEP_SECS"],
    # "DATABRICKS_TOKEN": os.environ["DATABRICKS_TOKEN"],
    # "DATABRICKS_HOST": os.environ["DATABRICKS_HOST"],
}

Reinicio del cache para que se tomen los cambios en el RAG

In [0]:
import sys, importlib

def unload_rag_modules():
    prefixes = ("rag_agent", "rag_core", "rag_lib")
    for m in list(sys.modules.keys()):
        if m in prefixes or any(m.startswith(p + ".") for p in prefixes):
            sys.modules.pop(m, None)
    importlib.invalidate_caches()

unload_rag_modules()

Variables para activar el trace

In [0]:
# # 1) Activar trace
os.environ["RAG_DEBUG_TRACE"] = "0"
os.environ["RAG_DEBUG_TRACE_ALL"] = "0"   # si querés todos los candidatos

Preguntas que parsa el smoketest

In [0]:
## Si no tiene comentarios al lado es porque se está contestando correctamente

# query = "¿Cuál es el dato de previsiones para octubre 2025?"
# query = "¿Cuáles son los valores de previsiones para octubre 2025 por segmento?"
# query = "¿Cuáles es el valor de previsiones para octubre 2025 para empresas?"
# query = "Cual es el retorno sobre activos para octubre de 2025?"
# query = "Cual es el retorno sobre patrimonio para octubre de 2025?"
# query = "Como ha sido la evolución del ROE?" #Mejorar obtención de info de gráficos
# query = "¿Como ha sido la evolución del ROE a lo largo del tiempo?"
# query = "Como ha sido la evolución del ROA?"

# query = "¿Cómo fue el resultado comercial sin ajuste por inflación en septiembre de 2025 y cuál ha sido el acumulado del año 2025?"
# query = "¿Cómo es el TNA para octubre 2024 para leasing?"
# query = "Cual fue el resultado comercial neto de corporate para 2025?"
# query = "Cual fue el resultado comercial neto AxI de corporate para 2025?"
# query = "Cuales fueron las previsiones de corporate por cada mes de 2025?"
# query = "Cual es la diferencia entre el acumulado de 2025 real vs el presupuestado?"
# query = "cuál es la variación entre el acumulado del Resultado Comercial Gestion Neto AXI para 2025 vs el presupuestado para 2025?"
# query = "cuál es la diferencia entre el Resultado Comercial Gestion Neto AXI acumulado para 2025 vs lo presupuestado para 2025?"

# query = "Calcula la diferencia porcentual entre el Resultado Comercial Neto AxI acumulado para 2025 vs lo presupuestado para 2025"
query = 'Dame el resultado neto para el cliente santander para julio 2025'




SMOKE TEST: Respuesta curada

In [0]:
import importlib.util
from pathlib import Path
from textwrap import shorten
import sys

# --- 1) Cargar el módulo ---
agent_py = Path("/Workspace/Users/emanuel.vieira@sunnydata.ai/bind_agent/src/rag/rag_agent.py")
sys.path.insert(0, str(agent_py.parent))  # para que importe rag_lib
print(sys.path[0])

spec = importlib.util.spec_from_file_location("rag_agent_module", str(agent_py))
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

# --- 2) Helpers de pretty print ---
def _pp_result(res: dict, max_hits: int = 5, max_chars: int = 280):
    print("\n" + "="*88)
    print("QUERY:", res.get("query",""))
    print("-"*88)
    print("ANSWER:\n")
    print(res.get("answer","").strip())
    print("-"*88)

    cands = res.get("retrieved_candidates") or []
    reranked = res.get("reranked_hits") or []

    print(f"Retrieved candidates: {len(cands)} | Reranked hits: {len(reranked)}")

    hits = reranked if reranked else cands
    if not hits:
        print("\n(No hay hits para mostrar)")
        return

    print(f"\nTOP {min(max_hits, len(hits))} EVIDENCIAS:")
    for i, h in enumerate(hits[:max_hits], 1):
        file_date = h.get("file_date") or ""
        path = h.get("path") or ""
        page = h.get("page_num")
        topic = h.get("topic") or ""
        cid = h.get("chunk_id") or ""

        loc = []
        if file_date: loc.append(file_date)
        if path: loc.append(path)
        if page is not None: loc.append(f"p.{page}")
        if topic: loc.append(f"topic={topic}")
        loc = " | ".join(loc)

        print(f"\n[{i}] {loc}")
        print(f"    chunk_id: {cid}")

def _pp_minimal(res: dict):
    # por si querés una versión ultra corta
    print("\n" + "="*88)
    print(res.get("answer","").strip())
    print("="*88)


In [0]:
res = mod.answer_with_rag(query)

# Elegí una:
_pp_result(res, max_hits=10, max_chars=500)
# _pp_minimal(res)

In [0]:
from rag_lib.business_rules import get_rules_snippet, get_relevant_rules

query = "¿Cómo fue el resultado comercial nominal sin ajuste por inflación en septiembre de 2025?"

# Ver qué reglas se detectan
rules = get_relevant_rules(query)
print("Reglas detectadas:")
for r in rules:
    print(f"  - {r.get('nombre')}")

# Ver el snippet completo
snippet = get_rules_snippet(query)
print("\nSnippet para el prompt:")
print(snippet[:1000])  # Primeros 1000 chars